# Protein Identification

From the diffential abundance analysis, some taxa were particularly enriched or depleated from the lifestyle groups. We decided to pick out 1 particulary enriched and 1 depleated taxa when comparing the the groups "Western" and "Fisher".

Here are the taxonomy with the respective feature ID:

bee4ccf1-9015-4cfb-82a3-105a91e5acb9 : d__Bacteria;k__Bacillati;p__Actinomycetota;c__Actinomycetes;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Bifidobacterium;s__Bifidobacterium_bifidum

10902617-4ae8-45aa-b3cd-fae31158d31b: d__Bacteria;k__Pseudomonadati;p__Pseudomonadota;c__Gammaproteobacteria;o__Aeromonadales;f__Succinivibrionaceae;g__Succinivibrio;s__Succinivibrio_dextrinosolvens


First let's setup the notebook.

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import qiime2 as q2
from qiime2 import Visualization

In [2]:
data_dir = 'updog_data'
os.makedirs(data_dir, exist_ok=True)

Creating the dataframe with the relevant feature IDs and save it as a tsv table:

In [8]:
df_ids = pd.DataFrame({'FeatureID': ['bee4ccf1-9015-4cfb-82a3-105a91e5acb9',
                                     '10902617-4ae8-45aa-b3cd-fae31158d31b']})


,FeatureID
0,bee4ccf1-9015-4cfb-82a3-105a91e5acb9
1,10902617-4ae8-45aa-b3cd-fae31158d31b


In [12]:
df_ids.to_csv(f'{data_dir}/ids.tsv', sep="\t", index=False)

## 1. Filter out the selected MAGs from the dereplicated MAGs file

We then select only the two feature IDs out of the dereplicated MAGs file, using the table we created before.

**the following command was run on Euler**

In [ ]:
! qiime feature-table filter-features \
    --i-table $data_dir/mags_derep_all_domains.qza \
    --m-metadata-file $data_dir/ids.tsv \
    --p-no-exclude-ids \
    --o-filtered-table $data_dir/mags-filtered-for-proteins.qza

Now we can import the generated file from the previous command for the next steps on the notebook:

In [13]:
!wget -O "$data_dir/mags-filtered-for-proteins.qza" "https://polybox.ethz.ch/index.php/s/sCy8P9MbW9TWXQr"

--2025-12-05 11:10:56--  https://polybox.ethz.ch/index.php/s/QMypfaA4G48CeHA
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30556 (30K) [text/html]
Saving to: ‘updog_data/mags-filtered-for-proteins.qza’

updog_data/mags-fil 100%[===================>]  29.84K  --.-KB/s    in 0s      

2025-12-05 11:10:57 (60.6 MB/s) - ‘updog_data/mags-filtered-for-proteins.qza’ saved [30556/30556]



In [17]:
! qiime tools peek $data_dir/mags-filtered-for-proteins.qza

UUID:        d519f1e5-5742-4667-b0fd-3fa7415811c1
Type:        FeatureData[MAG]
Data format: MAGSequencesDirFmt


## 2. Prediction of coded proteins

The `annotate predict-genes-prodigal` command scans a prokaryotic genome (or in this case, a MAG) and identifies all open reading frames that look like protein-coding genes.

In [18]:
! qiime annotate predict-genes-prodigal \
    --i-seqs $data_dir/mags-filtered-for-proteins.qza \
    --o-loci $data_dir/updog-loci.qza \
    --o-genes $data_dir/updog-genes.qza \
    --o-proteins $data_dir/updog-proteins.qza 

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved GenomeData[Loci] to: updog_data/updog-loci.qza
Saved GenomeData[Genes] to: updog_data/updog-genes.qza
Saved GenomeData[Proteins] to: updog_data/updog-proteins.qza


We then export the updog-proteins.qza file to obtain the fasta files of the protein preediction from both MAGs.

In [21]:
! qiime tools export \
  --input-path $data_dir/updog-proteins.qza  \
  --output-path $data_dir/updog-proteins

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported updog_data/updog-proteins.qza as ProteinsDirectoryFormat to directory updog-proteins


3. Import fasta files from relevant proteins

From [_this paper_](https://doi.org/10.1016/j.cub.2015.04.055) from Rampelli et al., we decided to focus on the proteins that were the most significantly different between the two populations they focused on. We then imported the fasta files of all the relevant proteins from the [NCBi protein database](https://www.ncbi.nlm.nih.gov/protein/). Once imported, we merge them into one file.

In [9]:
# Combine sequences into one file
!cat \
  $data_dir/fasta-enzymes/dehydratase.fasta \
  $data_dir/fasta-enzymes/formiminotransferase.fasta \
  $data_dir/fasta-enzymes/fructofuranosidase.fasta \
  $data_dir/fasta-enzymes/glucosidase.fasta \
  $data_dir/fasta-enzymes/xylosidase.fasta \
  > $data_dir/fasta-enzymes/all_enzymes.fasta

We then use the merged file and set it as our database.

In [10]:
!makeblastdb \
   -in $data_dir/fasta-enzymes/all_enzymes.fasta \
   -dbtype prot \
   -out $data_dir/updog_protein_db/updog_protein_db/



Building a new DB, current time: 12/05/2025 15:19:46
New DB name:   /home/jovyan/updog_perso/updog_data/updog_protein_db
New DB title:  updog_data/fasta-enzymes/all_enzymes.fasta
Sequence type: Protein
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 5 sequences in 0.0478439 seconds.




## 3. Blasting 

The fasta files from both relevant MAGs are then blasted with the database we previously created. The outputs are .txt files that are then compared with each other.

First for the enriched MAG in the "Western" group (depleted in the "Fisher" group).

In [16]:
!blastp \
   -query $data_dir/updog-proteins/10902617-4ae8-45aa-b3cd-fae31158d31b.fasta \
   -db $data_dir/updog_protein_db \
   -out $data_dir/results-10902617-4ae8-45aa-b3cd-fae31158d31b.txt \
   -evalue 1e-5 \
   -outfmt 6 

then for the enriched MAG in the "Fisher" group (depleted in the "Wester" group).

In [15]:
!blastp \
   -query $data_dir/updog-proteins/bee4ccf1-9015-4cfb-82a3-105a91e5acb9.fasta \
   -db $data_dir/updog_protein_db \
   -out $data_dir/results-bee4ccf1-9015-4cfb-82a3-105a91e5acb9.txt \
   -evalue 1e-5 \
   -outfmt 6 

Command line argument error: Argument "query". File is not accessible:  `updog_data/updog-proteins/bee4ccf1-9015-4cfb-82a3-105a91e5acb9.fasta.fasta'


## 4. Comparing our results

In [27]:
import pandas as pd

cols = [
    "qseqid","sseqid","pident","length","mismatch","gapopen",
    "qstart","qend","sstart","send","evalue","bitscore"
]

# load both BLAST tables
df1 = pd.read_csv(f"{data_dir}/results-10902617-4ae8-45aa-b3cd-fae31158d31b.txt", sep="\t", names=cols)
df2 = pd.read_csv(f"{data_dir}/results-bee4ccf1-9015-4cfb-82a3-105a91e5acb9.txt", sep="\t", names=cols)

# keep only the best hit per query for each MAG
df1_best = df1.sort_values("evalue").groupby("sseqid").first().reset_index()
df2_best = df2.sort_values("evalue").groupby("sseqid").first().reset_index()


# add MAG labels
df1_labeled = df1_best.copy()
df1_labeled["MAG"] = "MAG1"

df2_labeled = df2_best.copy()
df2_labeled["MAG"] = "MAG2"

# combine them into one table
combined = pd.concat([df1_labeled, df2_labeled], ignore_index=True)

combined

,sseqid,qseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,MAG
0,BAC14512.1,Vix9mzyce6AMxDAY9t4Ndi_16,40.842,546,311,8,9,552,1,536,5.190000e-141,410.0,MAG1
1,CBW74632.1,GS7JmysonrbuYaBhwDpMvS_8,46.635,208,92,5,165,367,16,209,5.350000e-58,177.0,MAG1
2,BAC14512.1,PfVxVHQoGBumLQXYQcQnQK_13,27.170,530,294,14,20,518,4,472,9.730000e-70,225.0,MAG2
3,CBW74632.1,Kok32GtTFnNJs8f5xunUgz_10,46.305,203,95,2,3,200,16,209,5.740000e-61,179.0,MAG2
4,VWA44305.1,Kok32GtTFnNJs8f5xunUgz_26,53.846,39,18,0,35,73,43,81,1.140000e-12,50.1,MAG2


In [30]:
combined.to_csv(f'{data_dir}/combined.tsv', sep="\t", index=False)